# Nested financial model: credit_pipeline
Same seed-42 data and outer temporal cohorts as the baseline.
Nested sklearn components remain estimator objects; conversion must preserve their behavior.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

raw = pd.read_csv("data/portfolio.csv")
assert not raw.empty, "Empty portfolio"
assert raw["loan_id"].is_unique, "Duplicate entity IDs"
horizon_months = 12
train_end_month = 202101
portfolio_rows = len(raw)
df = raw.loc[raw["followup_months"] >= horizon_months].copy()
assert not df.empty, "No complete outcome windows"
assert df["default_12m"].isin([0, 1]).all(), "Invalid target"

In [ ]:
train = df.loc[df["snapshot_month"] < train_end_month].copy()
validation = df.loc[df["snapshot_month"].between(202201, 202212)].copy()
test = df.loc[df["snapshot_month"].between(202401, 202412)].copy()
assert not train.empty, "Empty cohort"
assert not validation.empty, "Empty cohort"
assert not test.empty, "Empty cohort"
assert train["outcome_end_month"].max() < validation["snapshot_month"].min()
assert validation["outcome_end_month"].max() < test["snapshot_month"].min()

In [ ]:
numeric_columns = [
    "monthly_income",
    "existing_debt_payment",
    "loan_amount",
    "term_months",
    "interest_rate_bp",
    "utilization_pct",
    "late_payments_12m",
    "employment_months",
    "internal_score",
]
categorical_columns = [
    "employment_type",
    "loan_purpose",
    "origination_channel",
]
feature_columns = numeric_columns + categorical_columns
X_train = train[feature_columns].copy()
X_validation = validation[feature_columns].copy()
X_test = test[feature_columns].copy()
y_train = train["default_12m"].astype(int)
y_validation = validation["default_12m"].astype(int)
y_test = test["default_12m"].astype(int)

In [ ]:
preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric_columns,
        ),
        (
            "categorical",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_columns,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

In [ ]:
model = Pipeline(
    [
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(C=1.0, max_iter=300, random_state=42)),
    ]
)
model.fit(X_train, y_train)

In [ ]:
validation_scores = model.predict_proba(X_validation)[:, 1]
candidate_thresholds = np.array([0.15, 0.25, 0.35, 0.5, 0.65])
validation_alerts = validation_scores[:, None] >= candidate_thresholds[None, :]
validation_misses = ((y_validation.to_numpy()[:, None] == 1) & ~validation_alerts).sum(axis=0)
validation_false_alerts = ((y_validation.to_numpy()[:, None] == 0) & validation_alerts).sum(axis=0)
validation_costs = validation_misses * 12 + validation_false_alerts
threshold = float(candidate_thresholds[np.argmin(validation_costs)])

In [ ]:
test_scores = model.predict_proba(X_test)[:, 1]
test_predictions = (test_scores >= threshold).astype(int)

In [ ]:
matrix = confusion_matrix(y_test, test_predictions, labels=[0, 1])
metrics = {
    "roc_auc": round(float(roc_auc_score(y_test, test_scores)), 8),
    "average_precision": round(float(average_precision_score(y_test, test_scores)), 8),
    "brier": round(float(brier_score_loss(y_test, test_scores)), 8),
    "precision": round(float(precision_score(y_test, test_predictions, zero_division=0)), 8),
    "recall": round(float(recall_score(y_test, test_predictions, zero_division=0)), 8),
    "alert_rate": round(float(test_predictions.mean()), 8),
    "positive_rate": round(float(y_test.mean()), 8),
    "threshold": threshold,
    "true_negatives": int(matrix[0, 0]),
    "false_positives": int(matrix[0, 1]),
    "false_negatives": int(matrix[1, 0]),
    "true_positives": int(matrix[1, 1]),
    "cost_units": int(matrix[1, 0] * 12 + matrix[0, 1]),
}

In [ ]:
cohort_counts = {
    "train_rows": len(train),
    "validation_rows": len(validation),
    "test_rows": len(test),
    "excluded_followup_rows": portfolio_rows - len(df),
    "embargo_rows": len(df) - len(train) - len(validation) - len(test),
}
feature_names = preprocessor.get_feature_names_out().tolist()
summary = {"counts": cohort_counts, "metrics": metrics, "features": feature_names}
scored_test = test[["loan_id", "snapshot_month", "default_12m"]].assign(
    score=test_scores, prediction=test_predictions
)
fitted_model = model